In [ ]:
import transformers
import tokenizers
import accelerate

print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("Accelerate:", accelerate.__version__)

Transformers: 4.40.0
Tokenizers: 0.19.1
Accelerate: 0.29.3


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from transformers import AutoTokenizer

model_path = "/content/drive/MyDrive/NeuroTwin/models/internlm-xcomposer2d5-7b"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    trust_remote_code=True,
    use_fast=False
)

print("TYPE:", type(tokenizer))
print("CALLABLE:", callable(tokenizer))

TYPE: <class 'transformers_modules.internlm-xcomposer2d5-7b.tokenization_internlm2.InternLM2Tokenizer'>
CALLABLE: True


In [ ]:
import gc
import torch
from transformers import AutoModel

del model
gc.collect()
torch.cuda.empty_cache()

model = AutoModel.from_pretrained(
    model_path,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    device_map="cuda",
    attn_implementation="eager"
).eval()

model.tokenizer = tokenizer

print("Model loaded!")
print("Attention:", model.config.attn_implementation)
print("Tokenizer callable:", callable(model.tokenizer))

Set max length to 16384


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Model loaded!
Attention: eager
Tokenizer callable: True


In [ ]:
import torch
import os

image_path = "/content/learner_01_correct.png"

print("Image exists:", os.path.exists(image_path))

query = """
Describe only what is visually observable in this learner-interaction screenshot.
Do not infer cognitive load, confusion, stress, attention, emotion, or any other internal state.
"""

with torch.autocast(device_type="cuda", dtype=torch.float16):
    with torch.no_grad():
        response, history = model.chat(
            tokenizer,
            query=query,
            image=[image_path],
            history=[],
            do_sample=False,
            num_beams=3,
            use_meta=True
        )

print("\n=== IXC-2.5 SANITY CHECK ===\n")
print(response)

Image exists: True

=== IXC-2.5 SANITY CHECK ===

The learner-interaction screenshot shows a virtual laboratory environment designed for an educational activity. The central focus is a large red sphere, which is the object of selection for the task at hand, as indicated by the task prompt "Select the red sphere. Choose the red sphere to continue." The sphere is surrounded by various laboratory items, including a microscope, test tubes, and a molecular model, which suggests the setting is a chemistry or science lab. The environment is interactive, as evidenced by the "Activity Complete! Excellent! You selected the red sphere." message and the "Finish" button, indicating that the user has successfully completed the task and is prompted to finish the activity. The score and attempts are displayed, showing a score of 10 and 2 attempts, suggesting that the user has achieved a perfect score on this particular task. There is also a hint feature available, as indicated by the "Instructions" an

In [ ]:
images = [
    "/content/learner_01_correct.png",
    "/content/learner_02_incorrect.png",
    "/content/learner_03_repeated_error.png",
    "/content/learner_04_help.png",
    "/content/learner_05_complete.png",
]

query = """
Image1 <ImageHere>;
Image2 <ImageHere>;
Image3 <ImageHere>;
Image4 <ImageHere>;
Image5 <ImageHere>;

These five screenshots show a learner interacting with the same educational task over time.

Describe the learner's sequence of observable actions in order.

Specifically identify:
- correct selections,
- incorrect selections,
- repeated mistakes,
- help-related interaction,
- task completion.

Use only visually observable information from the screenshots.
Do not infer cognitive load, confusion, stress, attention, emotion,
or any other internal cognitive state.

Answer in this format:

Sequence:
1.
2.
3.
4.
5.

Repeated mistake:
Help-related behavior:
Completion:
"""

In [ ]:
with torch.autocast(device_type="cuda", dtype=torch.float16):
    with torch.no_grad():
        response_seq, history_seq = model.chat(
            tokenizer,
            query=query,
            image=images,
            history=[],

            # IMPORTANT MEMORY REDUCTIONS
            hd_num=4,
            num_beams=1,
            max_new_tokens=300,

            do_sample=False,
            top_p=1.0,
            use_meta=False
        )

print("\n=== IXC-2.5 SEQUENCE RESPONSE ===\n")
print(response_seq)


=== IXC-2.5 SEQUENCE RESPONSE ===

Sequence:
1. The learner selects a red sphere, which is correct, and the task progresses.
2. In the second screenshot, the learner makes an incorrect selection, as indicated by the "Incorrect" message. The sphere selected is not the red one required for the task.
3. The third screenshot shows the learner making another incorrect selection, leading to another "Incorrect" message.
4. The fourth screenshot indicates that the learner has opened help options but still made an incorrect selection, resulting in yet another "Incorrect" message.
5. Finally, the fifth screenshot shows the learner selecting the correct red sphere, which is confirmed as correct with a "Correct!" message.

Repeated mistake: The learner made multiple incorrect selections of objects other than the red sphere.
Help-related behavior: The learner interacted with the help options, but despite this, they initially continued to make incorrect selections.
Completion: The learner eventuall

In [ ]:
import json

step4_result = {
    "model": "InternLM-XComposer-2.5",
    "input_type": "5-screenshot interaction sequence",
    "screenshots": [
        "learner_01_correct.png",
        "learner_02_incorrect.png",
        "learner_03_repeated_error.png",
        "learner_04_help.png",
        "learner_05_complete.png"
    ],
    "response": response_seq
}

output_path = "/content/ixc_step4_results.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(step4_result, f, indent=2, ensure_ascii=False)

print("Saved:", output_path)

Saved: /content/ixc_step4_results.json


In [ ]:
def assign_behavioral_label(
    task_progress=False,
    incorrect_interaction=False,
    repeated_error=False,
    help_requested=False,
    inactivity_threshold_exceeded=False,
):
    """
    Convert observable learner-behavior evidence into one of the
    three NeuroTwin behavioral labels.

    Labels:
    - Normal Process
    - Needs Support
    - Repeated Error
    """

    if repeated_error:
        return "Repeated Error"

    if (
        incorrect_interaction
        or help_requested
        or inactivity_threshold_exceeded
    ):
        return "Needs Support"

    if task_progress:
        return "Normal Process"

    return "Unclassified"


examples = [
    {
        "example": 1,
        "observation": "Correct red sphere selected",
        "task_progress": True,
    },
    {
        "example": 2,
        "observation": "Incorrect blue cube selected",
        "incorrect_interaction": True,
    },
    {
        "example": 3,
        "observation": "Incorrect interaction repeated",
        "repeated_error": True,
    },
    {
        "example": 4,
        "observation": "Help interface opened",
        "help_requested": True,
    },
    {
        "example": 5,
        "observation": "Correct selection and activity completion",
        "task_progress": True,
    },
]


for example in examples:
    label = assign_behavioral_label(
        task_progress=example.get("task_progress", False),
        incorrect_interaction=example.get("incorrect_interaction", False),
        repeated_error=example.get("repeated_error", False),
        help_requested=example.get("help_requested", False),
        inactivity_threshold_exceeded=example.get(
            "inactivity_threshold_exceeded", False
        ),
    )

    print(
        f"Example {example['example']}: "
        f"{example['observation']} -> {label}"
    )

Example 1: Correct red sphere selected -> Normal Process
Example 2: Incorrect blue cube selected -> Needs Support
Example 3: Incorrect interaction repeated -> Repeated Error
Example 4: Help interface opened -> Needs Support
Example 5: Correct selection and activity completion -> Normal Process


In [ ]:
import os

print(os.listdir("/content"))

['.config', 'learner_01_correct.png', 'behavior_mapper.py', 'learner_04_help.png', '.ipynb_checkpoints', 'drive', 'ixc_step4_results.json', 'learner_03_repeated_error.png', 'learner_02_incorrect.png', 'learner_05_complete.png', 'model_evidence.json', 'sample_data']


In [ ]:
%cd /content
!python behavior_mapper.py

/content

Example 1 — Correct interaction
CogAgent label: Normal Process
IXC label:      Normal Process

Example 2 — Incorrect interaction
CogAgent label: Needs Support
IXC label:      Needs Support

Example 3 — Repeated incorrect interaction
CogAgent label: Repeated Error
IXC label:      Repeated Error

Example 4 — Help-seeking
CogAgent label: Normal Process
IXC label:      Needs Support

Example 5 — Activity completion
CogAgent label: Normal Process
IXC label:      Normal Process
